# 보간법 baseline

×2, ×3, ×4 로 줄였다가 Nearest / Bilinear / Bicubic / Lanczos 로 되돌려 PSNR·SSIM 을 잰다. GPU 불필요.

## 1. 데이터

In [ ]:
import urllib.request
LIB = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main/lib'
for m in ['sr_utils.py', 'sr_models.py', 'sr_losses.py']:
    urllib.request.urlretrieve(f'{LIB}/{m}', m)

from sr_utils import *

show_data()        # validation 2패치 + test 2구역

## 2. 보간 적용

In [ ]:
KERNELS = {'Nearest': cv2.INTER_NEAREST, 'Bilinear': cv2.INTER_LINEAR,
           'Bicubic': cv2.INTER_CUBIC,  'Lanczos': cv2.INTER_LANCZOS4}
SCALES = [2, 3, 4]

down = lambda hr, s: cv2.resize(hr, (hr.shape[1] // s, hr.shape[0] // s), interpolation=cv2.INTER_AREA)
up   = lambda lr, s, k: cv2.resize(lr, (lr.shape[1] * s, lr.shape[0] * s), interpolation=KERNELS[k])

for i, stem in enumerate(REPS['validation']):
    hr = pair('validation', stem)[1]
    zoom([(k, up(down(hr, 4), 4, k)) for k in KERNELS] + [('Target HR', hr)],
         center=[(180, 300), (300, 80)][i],
         title=f'validation {i + 1} — {stem}, x4')

## 3. 정량 평가

In [ ]:
HR = {s: pair('validation', s)[1] for s in list_split('validation')}
res = {}
for s in SCALES:
    for k in KERNELS:
        v = [score(up(down(hr, s), s, k), hr) for hr in HR.values()]
        res[(s, k)] = (np.mean([x[0] for x in v]), np.mean([x[1] for x in v]))

print(f'{"":8s}' + ''.join(f'{k:>20s}' for k in KERNELS))
print(f'{"":8s}' + ''.join(f'{"PSNR":>10s}{"SSIM":>10s}' for _ in KERNELS))
for s in SCALES:
    print(f'x{s:<7d}' + ''.join(f'{res[(s,k)][0]:10.2f}{res[(s,k)][1]:10.4f}' for k in KERNELS))

json.dump({f'x{s}': {k: {'psnr': round(res[(s,k)][0], 3), 'ssim': round(res[(s,k)][1], 4)}
                     for k in KERNELS} for s in SCALES},
          open('baseline_interpolation.json', 'w'), indent=1)

In [ ]:
x, w = np.arange(len(SCALES)), 0.2
colors = ['#c96a5b', '#d9a441', '#2f6f9f', '#4f9d69']
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for j, (name, unit) in enumerate([('PSNR', ' (dB)'), ('SSIM', '')]):
    for i, k in enumerate(KERNELS):
        ax[j].bar(x + (i - 1.5) * w, [res[(s, k)][j] for s in SCALES], w, label=k, color=colors[i])
    ax[j].set_xticks(x); ax[j].set_xticklabels([f'x{s}' for s in SCALES])
    ax[j].set_title(f'{name}{unit} by scale and kernel')
    lo = min(res[(s, k)][j] for s in SCALES for k in KERNELS)
    hi = max(res[(s, k)][j] for s in SCALES for k in KERNELS)
    ax[j].set_ylim(lo - (hi - lo) * .15, hi + (hi - lo) * .15)
    ax[j].grid(axis='y', alpha=.3); ax[j].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 4. 결과

In [ ]:
for i in range(len(TESTS)):
    t = load_test(i)
    zoom([(k, up(t, 3, k)) for k in ['Nearest', 'Bicubic', 'Lanczos']],
         center=(800, 800), size=70,
         title=f'test {i + 1} (Incheon), x3 - no target')